In [1]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.decomposition import TruncatedSVD
import os
import wandb

from util.preprocessing import TweetPreprocessor 
import joblib

# Consts

In [3]:
MODEL_DIR = os.path.join(os.getcwd(), 'models')
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')

# Dataset

In [4]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
test_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_test.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))

KeyboardInterrupt: 

In [ ]:
x_train, y_train = train_df["text"], train_df["gender_label"]
x_test, y_test = test_df["text"], test_df["gender_label"]
x_val, y_val = val_df["text"], val_df["gender_label"]

# Initiate pipeline

In [ ]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
    ])),
    ("svd", TruncatedSVD()),
    ("clf", LinearSVC()),
])

# Init wandb

In [ ]:
wandbToken = "YOUR_WANDB_TOKEN"
wandb.login(key=wandbToken)

# Randomized search

In [ ]:
wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search",group="random_search")

ngram_ranges_word = [(1, 2), (1, 3), (2, 3)]
ngram_ranges_char = [(2, 4), (3, 5), (4, 6), (2, 5)]

rnd_params = {
    # word TF-IDF
    "features__tfidf_word__use_idf": [True, False],
    "features__tfidf_word__sublinear_tf": [True, False],
    "features__tfidf_word__norm": ["l1", "l2"],
    "features__tfidf_word__max_df": np.linspace(0.6, 0.9, 50).tolist(),
    "features__tfidf_word__min_df": np.linspace(0.001, 0.05, 50).tolist(),
    "features__tfidf_word__max_features": list(range(5000, 60001, 5000)),
    "features__tfidf_word__ngram_range": ngram_ranges_word,

    # char TF-IDF
    "features__tfidf_char__use_idf": [True, False],
    "features__tfidf_char__sublinear_tf": [True, False],
    "features__tfidf_char__norm": ["l1", "l2"],
    "features__tfidf_char__max_df": np.linspace(0.6, 0.9, 50).tolist(),
    "features__tfidf_char__min_df": np.linspace(0.001, 0.05, 50).tolist(),
    "features__tfidf_char__max_features": list(range(5000, 60001, 5000)),
    "features__tfidf_char__ngram_range": ngram_ranges_char,

    # SVD
    "svd__n_components": [10, 50, 100, 200, 300, 500, 600],

    # LinearSVC
    "clf__C": np.logspace(-2, 2, 50).tolist(),
}

In [ ]:
search = RandomizedSearchCV(
    pipeline,
    rnd_params,
    n_iter=500,
    cv=StratifiedKFold(n_splits=4),
    scoring="f1_macro",
    n_jobs=4,
    verbose=2,
    random_state=1930912391,
)

In [ ]:
search.fit(x_train, y_train) # type: ignore
for i, row in search.cv_results_.items():
    wandb.log({
        "type" : "random_search",
        "iteration": i,
        "mean_test_score": row['mean_test_score'],
        "std_test_score": row['std_test_score'],
        **{k: v for k, v in row["params"]}
    })

wandb.log({"best_score": search.best_score_, "best_params": search.best_params_})
model_path_rnd = joblib.dump(search.best_estimator_, os.path.join(MODEL_DIR, "rnd_model.joblib"))
if model_path_rnd is not None and os.path.exists(model_path_rnd.pop()):
    print(f"Model saved to {model_path_rnd[0]}")
    artifact = wandb.Artifact("best_model", type="model")
    artifact.add_file(model_path_rnd.pop())
    wandb.log_artifact(artifact)
    wandb.log_artifact(artifact, "random_search_best_model")
best_rnd = search.best_params_

# Grid search

In [ ]:
wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search",group="grid_search")

def make_range(value, *_args, pct=0.05, clip_min=None, clip_max=None, as_int=False):
    # 4 candidates around `value`: -10%, -5%, +5%, +10%
    factors = [1 - 2 * pct, 1 - pct, 1 + pct, 1 + 2 * pct]
    candidates = [value * f for f in factors]

    if clip_min is not None:
        candidates = [max(clip_min, c) for c in candidates]
    if clip_max is not None:
        candidates = [min(clip_max, c) for c in candidates]

    if as_int:
        candidates = [int(round(c)) for c in candidates]

    # remove duplicates while preserving order
    uniq = []
    for c in candidates:
        if c not in uniq:
            uniq.append(c)
    return uniq

grid_params = {
    # word TF-IDF
    "features__tfidf_word__use_idf":     [best_rnd["features__tfidf_word__use_idf"]],
    "features__tfidf_word__sublinear_tf":[best_rnd["features__tfidf_word__sublinear_tf"]],
    "features__tfidf_word__norm":        [best_rnd["features__tfidf_word__norm"]],
    "features__tfidf_word__ngram_range": [best_rnd["features__tfidf_word__ngram_range"]],
    "features__tfidf_word__max_df":      make_range(best_rnd["features__tfidf_word__max_df"]),
    "features__tfidf_word__min_df":      make_range(best_rnd["features__tfidf_word__min_df"]),
    "features__tfidf_word__max_features":make_range(best_rnd["features__tfidf_word__max_features"]),

    # char TF-IDF
    "features__tfidf_char__use_idf":     [best_rnd["features__tfidf_char__use_idf"]],
    "features__tfidf_char__sublinear_tf":[best_rnd["features__tfidf_char__sublinear_tf"]],
    "features__tfidf_char__norm":        [best_rnd["features__tfidf_char__norm"]],
    "features__tfidf_char__ngram_range": [best_rnd["features__tfidf_char__ngram_range"]],
    "features__tfidf_char__max_df":      make_range(best_rnd["features__tfidf_char__max_df"]),
    "features__tfidf_char__min_df":      make_range(best_rnd["features__tfidf_char__min_df"]),
    "features__tfidf_char__max_features":make_range(best_rnd["features__tfidf_char__max_features"]),

    # SVD
    "svd__n_components": [best_rnd["svd__n_components"]],

    # LinearSVC
    "clf__C": make_range(best_rnd["clf__C"], make_range(np.logspace(-2, 2, 50).tolist(), best_rnd["clf__C"]))
}

In [ ]:
grid_search = GridSearchCV(
    pipeline,
    grid_params,
    cv=StratifiedKFold(n_splits=4),
    scoring="f1_macro",
    n_jobs=4,
    verbose=2,
)

In [ ]:
grid_search.fit(x_train.to_list(), y_train.to_list(66))

results = grid_search.cv_results_

for i in range(len(results["params"])):
    wandb.log({
        "type": "grid_search",
        "iteration": i,
        "mean_test_score": results["mean_test_score"][i],
        "std_test_score": results["std_test_score"][i],
        **results["params"][i]
    })

model_path = os.path.join(MODEL_DIR, "grid_model.joblib")
joblib.dump(grid_search.best_estimator_, model_path)

artifact = wandb.Artifact("best_model_grid", type="model")
artifact.add_file(model_path)
wandb.log_artifact(artifact)

wandb.log({
    "grid_best_score": grid_search.best_score_,
    **{f"best_{k}": v for k, v in grid_search.best_params_.items()}
})